# Anthropic Batch Runs


## Setup


In [ ]:
# Libraries
import json
import sys
from pathlib import Path
import pandas as pd

In [ ]:
# Project Root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
# Pipeline

%load_ext autoreload
%autoreload 2

import backends
import run
import settings
import utils

# Pipeline Check


needs = {'run': ['write_batch', 'read_batch', 'batch_path', 'set_aside_replies'],
         'utils': ['api_key', 'read_lines', 'read_table', 'result_path',
                   'model_slug', 'make_directories'],
         'backends': ['USAGE', 'spent', 'record_usage', 'takes_sampling'],
         'settings': ['BATCHES_DIR', 'MODELS', 'GENERATION']}
missing = [f'{name}.{attr}' for name, attrs in needs.items()
           for attr in attrs if not hasattr(globals()[name], attr)]
if missing:
    raise SystemExit('Scripts are out of date, missing: ' + ', '.join(missing)
                     + '\nCopy scripts/ from the latest package and restart the kernel.')

utils.make_directories()
pd.set_option('display.max_colwidth', 70)
print('Scripts are current')

## Model


In [ ]:
MODEL = 'claude-haiku-4-5-20251001'

spec = next(e for e in settings.MODELS.values() if e['id'] == MODEL)
if spec['provider'] != 'anthropic':
    raise SystemExit(f'{MODEL} is served by {spec["provider"]}, not Anthropic')

prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']
have = len(utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR)))

print(f'Model      {MODEL}')
print(f'Billed at  ${spec["price"]["input"]}/M input, '
      f'${spec["price"]["output"]}/M output, halved on a batch')
print(f'Sampling   {"as the design asks" if backends.takes_sampling(MODEL) else "provider defaults"}, '
      f'temperature {settings.GENERATION["temperature"]}')
print(f'Cap        {settings.GENERATION["max_tokens"]} tokens')
print(f'Collected  {have:,} of {wanted:,}')
print(f'Key found  {bool(utils.api_key("anthropic"))}')

## Rerun


In [ ]:
FRESH = False        # Fresh Run

if FRESH:
    moved = run.set_aside_replies(MODEL)
    print(f'Earlier pass set aside at {moved}' if moved
          else 'Nothing collected yet, so nothing to set aside')
else:
    print('Normal run: only what is missing will be requested')

## Requests


In [ ]:
path, count = run.write_batch(MODEL)

if path is None:
    print('Nothing outstanding for this model')
else:
    print(f'{count:,} requests written to {path}')
    print()
    print(json.dumps(json.loads(path.read_text().splitlines()[0]), indent=2))

## Cost


In [ ]:
OUTPUT_TOKENS = 308         
INPUT_TOKENS = 24

if path is None:
    print('Nothing to price, this model is already collected')
else:
    price = spec['price']
    standard = (count * INPUT_TOKENS * price['input']
                + count * OUTPUT_TOKENS * price['output']) / 1e6
    print(f'{count:,} calls at {OUTPUT_TOKENS} output tokens each')
    print(f'  Standard  ${standard:,.2f}')
    print(f'  Batched   ${standard / 2:,.2f}')

## Submit


In [ ]:
from anthropic import Anthropic

client = Anthropic(api_key=utils.api_key('anthropic'))

requests = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
job = client.messages.batches.create(requests=requests)

job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
job_file.parent.mkdir(parents=True, exist_ok=True)
job_file.write_text(job.id)
print(f'Submitted {job.id}, {job.processing_status}')

print(f'Requests kept at {run.name_after_job(MODEL, job.id)}')

## Resume Job


In [ ]:
from anthropic import Anthropic

client = Anthropic(api_key=utils.api_key('anthropic'))

jobs = [{'id': b.id, 'status': b.processing_status,
         'succeeded': b.request_counts.succeeded,
         'errored': b.request_counts.errored,
         'processing': b.request_counts.processing,
         'created': pd.to_datetime(str(b.created_at))}
        for b in client.messages.batches.list(limit=10).data]
display(pd.DataFrame(jobs))

In [ ]:
# Select Job
ADOPT = jobs[0]['id'] if 'jobs' in dir() and jobs else ''

if ADOPT:
    job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
    job_file.parent.mkdir(parents=True, exist_ok=True)
    job_file.write_text(ADOPT)
    print(f'Adopted {ADOPT} for {MODEL}')

## Status


In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')

if not job_file.exists():
    print('No job submitted for this model yet, run the cell above')
else:
    job = client.messages.batches.retrieve(job_file.read_text().strip())
    counts = job.request_counts
    total = (counts.succeeded + counts.errored + counts.canceled
             + counts.expired + counts.processing) or 1
    print(f'Job {job.id}')
    print(f'{job.processing_status.capitalize()}, {counts.succeeded:,} succeeded, '
          f'{counts.errored} errored, {counts.processing:,} still processing '
          f'({counts.succeeded / total:.0%})')

## Results


In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
job = (client.messages.batches.retrieve(job_file.read_text().strip())
       if job_file.exists() else None)

if job is None or job.processing_status != 'ended':
    print(f'Nothing to read yet: '
          f'{job.processing_status if job else "no job adopted"}')
else:
    results = run.batch_path(MODEL, 'output', job.id)
    with results.open('w') as file:
        for entry in client.messages.batches.results(job.id):
            file.write(json.dumps(entry.model_dump(mode='json')) + '\n')
    print(f'Wrote {results}')

    backends.USAGE.update(calls=0, input=0, output=0)
    read, failed, truncated, repeated, blocked = run.read_batch(MODEL, results)

    usage, cost = backends.USAGE, backends.spent(MODEL)
    print(f'\nRead {read:,} replies, {failed} failed, {truncated} truncated, '
          f'{repeated:,} already had')
    print(f'Tokens: {usage["input"]:,} input, {usage["output"]:,} output')
    print(f'Cost: ${cost:,.2f} standard, ${cost / 2:,.2f} batched')
    print(f'Output tokens a reply: '
          f'{usage["output"] / max(read - failed, 1):.0f}')

## Validation


In [ ]:
collected = utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR))
prompts = utils.read_table(settings.PROMPTS_PATH)

if collected.empty:
    print(f'Nothing collected for {MODEL} yet')
else:
    blank = int((collected['response'].astype(str).str.strip() == '').sum())
    errored = int((collected['error'].astype(str).str.strip() != '').sum())
    print(f'Replies: {len(collected):,}, {blank} empty, {errored} errored')
    print(f"Coverage: {collected['prompt_id'].nunique():,} of {len(prompts):,} "
          f"prompts")

    display(collected.merge(prompts[['prompt_id', 'condition', 'prompt']],
                            on='prompt_id')[['condition', 'prompt',
                                             'response']].head(5))